In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib, subprocess, sys
REPO_URL='https://github.com/RICHAAARC/CEG-WM.git'; BRANCH='Geometry-V4'
SOURCE_EXACT='5b29a275151d436dbe1d51789cffe8e6908966b7'
REPO=Path('/content/cegwm-geometry-v4-g0-g1-source'); DRIVE_RUNS=Path('/content/drive/MyDrive/CEG-WM/Geometry-V4-G0-G1/runs')
if REPO.exists(): raise FileExistsError('fresh runtime required')
subprocess.run(['git','clone','--single-branch','--branch',BRANCH,REPO_URL,str(REPO)],check=True)
def git(*args): return subprocess.run(['git',*args],cwd=REPO,check=True,capture_output=True,text=True).stdout.strip()
subprocess.run(['git','checkout','--detach',SOURCE_EXACT],cwd=REPO,check=True)
assert git('rev-parse','HEAD')==SOURCE_EXACT and git('branch','--show-current')=='' and git('status','--porcelain')==''
CONFIG_SHA256=hashlib.sha256((REPO/'configs/geometry_v4/geometry_v4_g0_g1_v1.json').read_bytes()).hexdigest(); RUN_ROOT=DRIVE_RUNS/f'{SOURCE_EXACT}-{datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")}'
DRIVE_RUNS.mkdir(parents=True,exist_ok=True)
if RUN_ROOT.exists(): raise FileExistsError('create-only run exists')


In [ ]:
import os, torch
from google.colab import userdata
assert torch.cuda.is_available(), 'GPU required; no G0 record on CPU'
subprocess.run([sys.executable,'-m','pip','install','-q',str(REPO)],check=True)
root_key=userdata.get('CEG_WM_ROOT_KEY'); hf_token=userdata.get('HF_TOKEN'); assert all(isinstance(x,str) and x.strip() for x in (root_key,hf_token))
markers=('TOKEN','KEY','SECRET','PASSWORD','CREDENTIAL'); runner_env={k:v for k,v in os.environ.items() if not any(m in k.upper() for m in markers)}; runner_env['CEG_WM_ROOT_KEY']=root_key; runner_env['HF_TOKEN']=hf_token; root_key=''; hf_token=''
command=[sys.executable,'-m','experiments.geometry_v4_generative_engine','--stage','G0','--repo-root',str(REPO),'--artifact-root',str(RUN_ROOT),'--expected-exact',SOURCE_EXACT]
try: completed=subprocess.run(command,cwd=REPO,env=runner_env,text=True,stdout=subprocess.PIPE,stderr=subprocess.DEVNULL,check=False)
finally: runner_env.pop('CEG_WM_ROOT_KEY',None); runner_env.pop('HF_TOKEN',None); runner_env=None
summary=completed.stdout[-8192:]
if completed.returncode!=0: raise RuntimeError('Geometry-V4 G0 subprocess stopped: '+summary)
print(summary)
